#### Setup & Data Loading

In [1]:
import pandas as pd
import os

# Define paths
RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

# Load the 3 different formats
df_cma = pd.read_excel(os.path.join(RAW_DIR, "CMA_CGM_Jan2025_Portal.xlsx"))
df_msc = pd.read_csv(os.path.join(RAW_DIR, "MSC_Export_Log_Jan2025.csv"))
df_maersk = pd.read_json(os.path.join(RAW_DIR, "Maersk_API_Feed_Jan2025.json"))

print(f"Loaded: CMA ({len(df_cma)}), MSC ({len(df_msc)}), Maersk ({len(df_maersk)})")

Loaded: CMA (73), MSC (58), Maersk (49)


#### The "Normalization" Function

In [2]:
def normalize_carrier_data(df, carrier_name):
    # Mapping dictionary to fix the naming chaos
    mapping = {
        'BookingID': 'Booking_Ref',
        'Shipment_Ref': 'Booking_Ref',
        'Verified_Weight': 'Actual_VGM_KG',
        'Port_Scale_KG': 'Actual_VGM_KG'
    }
    
    # Rename columns based on the carrier's specific messy headers
    df = df.rename(columns=mapping)
    
    # Add a standard 'Carrier' column if it's missing or messy
    df['Carrier'] = carrier_name
    
    return df

# Apply normalization
clean_cma = normalize_carrier_data(df_cma, "CMA CGM")
clean_msc = normalize_carrier_data(df_msc, "MSC")
clean_maersk = normalize_carrier_data(df_maersk, "Maersk")

# Merge all into one Master DataFrame
df_unified = pd.concat([clean_cma, clean_msc, clean_maersk], ignore_index=True)

print("Unified Romina Logistics Database Created.")
df_unified.head()

Unified Romina Logistics Database Created.


,Booking_Ref,Carrier,Origin_Inland,Port_of_Loading,Destination,Grade,Declared_Weight_KG,Actual_VGM_KG,Booking_Date,SI_Deadline
0,ROM25-18965,CMA CGM,Modjo Hub,Djibouti,Trieste (Italy),Guji G1 Specialty,19204.96,19204.96,2025-01-01,2025-01-06
1,ROM25-59754,CMA CGM,Mojo Dry Port,Djibouti,Trieste (Italy),Sidama G1 Washed,19210.90,19210.90,2025-01-02,2025-01-07
2,ROM25-16743,CMA CGM,Modjo Hub,Djibouti,Hamburg (Germany),Sidama G1 Washed,19204.84,19204.84,2025-01-03,2025-01-08
3,ROM25-18151,CMA CGM,Modjo Hub,Djibouti,Trieste (Italy),Guji G1 Specialty,19204.56,19204.56,2025-01-03,2025-01-08
4,ROM25-15342,CMA CGM,Mojo Dry Port,Djibouti,Jeddah (KSA),Yirgacheffe G2 Natural,19205.56,19205.56,2025-01-05,2025-01-10


#### The Intelligence Layer (Flagging the 17% Errors)

In [3]:
# Calculate the Weight Variance
df_unified['Weight_Variance'] = df_unified['Actual_VGM_KG'] - df_unified['Declared_Weight_KG']

# Flag any container where the variance is > 50kg (Standard maritime tolerance)
df_unified['VGM_Alert'] = df_unified['Weight_Variance'].apply(lambda x: "CRITICAL: ROLL RISK" if abs(x) > 50 else "CLEAR")

# Calculate potential loss prevented
# Assuming each 'Rolled' container costs Romina $5,000 in demurrage and quality loss
rolls_prevented = len(df_unified[df_unified['VGM_Alert'] != "CLEAR"])
potential_savings = rolls_prevented * 5000

print(f"ANALYSIS COMPLETE")
print(f"Total Containers Screened: {len(df_unified)}")
print(f"Critical VGM Errors Detected: {rolls_prevented}")
print(f"Potential Revenue Protected: ${potential_savings:,}")

# Save the 'Action List' for the Senior Officer
df_unified.to_csv(os.path.join(PROCESSED_DIR, "Unified_Logistics_Final_2025.csv"), index=False)

ANALYSIS COMPLETE
Total Containers Screened: 180
Critical VGM Errors Detected: 31
Potential Revenue Protected: $155,000
